# Marker Tutorial: Fast PDF to Markdown Conversion in Python
Marker is one of the fastest open source PDF to Markdown converters, tuned for speed and clean LLM ready output. A hands on tutorial with install steps, working code, and the tradeoffs versus MinerU and PyMuPDF4LLM.

Marker is one of the most popular open source tools for turning PDFs into clean Markdown. It was built with one goal in mind: produce accurate, structured output fast, without the heavy per page cost of a full document AI pipeline. If you have a folder of PDFs and you want Markdown that a large language model can actually read, Marker is a strong first choice.

This tutorial walks through installing Marker, converting your first document in Python, batching a whole directory, and tuning the settings that matter. It also covers the honest tradeoffs versus MinerU and PyMuPDF4LLM so you can pick the right tool instead of the loudest one. Everything here is code you can copy, paste, and run today.

## What Marker Actually Does
Marker is a Python library that runs a pipeline of deep learning models over a PDF. It detects the layout of each page, orders the reading flow, recognizes text, and reconstructs the document as Markdown. Unlike a plain text extractor, it understands that a page has headings, paragraphs, lists, tables, code blocks, and figures, and it emits Markdown that preserves that structure.

A few things make Marker stand out:

* **Speed with GPU**: On a modern GPU, Marker processes pages far faster than most vision heavy alternatives, which makes it practical for large batches.
* **Clean output**: The Markdown is tuned for downstream use in LLM pipelines, with sensible heading levels and preserved tables.
* **Formats beyond PDF**: Recent versions also handle formats like DOCX, PPTX, and EPUB, so it can be a single entry point for mixed document sets.
* **OCR when needed**: Marker runs OCR on scanned or image based pages, so it does not fall over on documents that have no embedded text layer.
* **Optional LLM boost**: You can layer in a language model to improve accuracy on tricky tables and inline math, at the cost of extra time and API calls.

The catch is that Marker is model based, so it wants Python 3.10 or newer, a decent amount of RAM, and ideally a GPU. On CPU only machines it still works, but it is slower. If you just need Markdown from one file without setting up an environment, an online converter like pdftomd.cloud does the same job in your browser in seconds.

## Installing Marker

In [ ]:
from weasyprint.css.validation.properties import display
%python -m venv marker-env
%source marker-env/bin/activate   # Windows: marker-env\Scripts\activate

%pip install marker-pdf

# For non PDF formats (DOCX, PPTX, EPUB, and more)
%pip install "marker-pdf[full]"

The first time you run a conversion, Marker downloads its model weights. That download is a one time cost of a few hundred megabytes, so the very first run is slower than every run after it. If you are deploying to a server or a container, warm the cache during the build so production requests are not stuck waiting on a download.

You will need a [llama-server binary](https://github.com/ggml-org/llama.cpp/releases) too. Install with:
  * macOS:
    > brew install llama.cpp
  * Linux:
    > brew install llama.cpp  OR download from

Or set LLAMA_CPP_BINARY in your env to the binary path.

## Your First Conversion in Python
The core of Marker is a converter object that you build once and then call on a file path. Here is the minimal script to turn a single PDF into a Markdown string and write it to disk.

In [5]:
import json
import os

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered, save_output

output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)

# Load the models once. This is the expensive step, so reuse it.
converter = PdfConverter(artifact_dict=create_model_dict())

# Run the conversion.
rendered = converter("../../data/llama_research_paper.pdf")
markdown, metadata, images = text_from_rendered(rendered)

# Save the Markdown text file
print(f"Wrote {len(markdown)} characters of Markdown")
with open("outputs/report.md", "w", encoding="utf-8") as f:
    f.write(markdown)

# Save any extracted images
print(f"Save {len(images)} extracted images")
for filename, img in images.items():
    img.save(os.path.join(output_dir, filename))

# The object directly exposes a .metadata dictionary
metadata_dict = rendered.metadata if len(metadata) < 3 else metadata

# Save the metadata as a JSON file
with open(os.path.join(output_dir, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata_dict, f, indent=4, ensure_ascii=False)

2026-11-09 12:53:09,183 [INFO] marker: Table processing stats: {'tables_pdftext': 15, 'tables_ocr': 2, 'tables_total': 17}


Wrote 109469 characters of Markdown


## Improving Accuracy with the LLM Option
Marker can call a language model to clean up its output on the hardest parts of a document: merged table cells, inline math, and ambiguous layouts. You enable it by passing a flag and supplying an API key for the provider you want to use.

In [49]:
import json

from pathlib import Path
from tqdm import tqdm

from dotenv import load_dotenv
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser
from marker.output import text_from_rendered

load_dotenv()

# --- CONFIG ---
selected_config = {
    "use_llm": True,
    "output_format": "markdown",
    "llm_service": "marker.services.ollama.OllamaService",
    "ollama_base_url": "http://localhost:11434",
    "ollama_model": "gemma3:4b",
}

source = Path("../../data")
target = Path("./outputs")
target.mkdir(exist_ok=True)

# In sequential mode, we load the converter ONCE.
# This is MUCH faster than loading it inside a loop.
print("Loading Models... (This may take a minute)")
config_parser = ConfigParser(selected_config)
converter = PdfConverter(
    artifact_dict=create_model_dict(),
    config=config_parser.generate_config_dict(),
    llm_service=config_parser.get_llm_service(),
)

pdf_files = sorted(list(source.glob("R*.pdf")))

# Use tqdm for a beautiful notebook progress bar
for file in tqdm(pdf_files, desc="Processing PDFs"):
    try:
        rendered = converter(str(file))
        markdown, metadata, images = text_from_rendered(rendered)

        file_target_dir = target / file.stem
        file_target_dir.mkdir(parents=True, exist_ok=True)

        # Save Markdown
        (file_target_dir / f"{file.stem}.md").write_text(markdown, encoding="utf-8")

        # Save Images
        for filename, img in images.items():
            img.save(file_target_dir / filename)

        # Save Metadata
        metadata_dict = rendered.metadata if len(metadata) < 3 else metadata
        with open(file_target_dir / f"{file.stem}_metadata.json", "w", encoding="utf-8") as f:
            json.dump(metadata_dict, f, indent=4, ensure_ascii=False)

    except Exception as err:
        print(f"❌ Failed on {file.name}: {err}")

print("Done!")

Loading Models... (This may take a minute)


Processing PDFs:   0%|          | 0/1 [00:00<?, ?it/s]2026-11-09 15:44:16,458 [INFO] marker: Table processing stats: {'tables_pdftext': 2, 'tables_ocr': 3, 'tables_total': 5}

LLMTableProcessor running: 100%|██████████| 5/5 [01:02<00:00, 12.55s/it]

LLM processors running:  44%|████▍     | 4/9 [00:31<00:32,  6.52s/it]2026-11-09 15:47:16,188 [WARNING] marker: Ollama inference failed: Unterminated string starting at: line 1 column 14 (char 13)

LLM processors running: 100%|██████████| 9/9 [02:25<00:00, 16.19s/it]

Running LLMSectionHeaderProcessor:   0%|          | 0/1 [00:00<?, ?it/s]2026-11-09 15:47:45,042 [WARNING] marker: Ollama inference failed: 400 Client Error: Bad Request for url: http://localhost:11434/api/generate
2026-11-09 15:47:45,044 [WARNING] marker: LLM did not return a valid response
Processing PDFs: 100%|██████████| 1/1 [06:40<00:00, 400.06s/it]

Done!


A few things worth noting. The create_model_dict() call loads every model Marker needs into memory. It is slow, so build the converter one time and reuse it across many files rather than rebuilding it in a loop. The text_from_rendered helper unpacks the rendered result into the Markdown string, a metadata dictionary, and any extracted images. If your document has figures you want to keep, save those images alongside the Markdown so the links resolve.

## Converting from the Command Line
If you do not want to write Python at all, Marker exposes two commands. Use marker_single for one file and marker for a whole folder.
```bash
# Convert a single file into an output directory
marker_single report.pdf --output_dir ./out

# Convert every PDF in a folder
marker ./pdfs --output_dir ./out --workers 4
```
The --workers flag controls how many documents run in parallel. Set it based on your GPU memory: more workers finish a batch faster but each one holds its own copy of the working data. Start with a small number, watch your memory, and increase it until you find the ceiling. For a deeper look at folder wide conversion patterns, see our guide on batch converting PDFs from the command line.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from dotenv import load_dotenv
from functools import partial
from marker_worker import process_pdf_worker
from pathlib import Path
from tqdm import tqdm

load_dotenv()

# Config
selected_config = {
    "use_llm": True,
    "output_format": "markdown",
    "llm_service": "marker.services.ollama.OllamaService",
    "ollama_base_url": "http://localhost:11434",
    "ollama_model": "gemma3:4b",
}

source = Path("../../data")
target = Path("./outputs")
target.mkdir(exist_ok=True)
pdf_files = sorted(list(source.glob("*.pdf")))

# MAX_WORKERS: Keep this very low on Mac!
# 2 workers = 2 sets of layout models + Ollama
MAX_WORKERS = 2

print(f"Launching {MAX_WORKERS} workers via marker_worker.py...")

# We use partial to pass the config and target path
worker_func = partial(process_pdf_worker, config_dict=selected_config, target_root=str(target))

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Using tqdm to track progress
    results = list(tqdm(executor.map(worker_func, [str(f) for f in pdf_files]), total=len(pdf_files)))

for res in results:
    print(res)


Turning on use_llm noticeably improves tables and equations, which is exactly where most converters struggle. The tradeoff is speed and cost: each page that needs help triggers an API call, so a large batch adds up. A good rule is to run without the LLM by default and only switch it on for documents where table fidelity is critical, such as financial reports or data sheets. If tables are your main pain point, our guide on [converting PDF tables](https://pdftomd.cloud/blog/convert-pdf-tables-to-markdown) to clean Markdown covers the failure modes in depth.